In [5]:
"""
Online Distillation — DML (Deep Mutual Learning)
=================================================
Method  : DML — Deep Mutual Learning (Zhang 2018)
Models  : ResNet-50 × 2 peers (no pre-trained teacher)
Dataset : CIFAR-10
Config  : τ=3.0, warm-start init, cosine LR, AMP, 10 epochs

DML LOSS:
  Each model i minimises:
    L_i = CE(y, p_i) + (1/(N-1)) * Σ_{j≠i} KL(p_j || p_i)
  All peer KL terms use detached probabilities → only model i's graph
  is active during model i's backward pass.

WARM-START:
  Each peer is pre-trained for WARMUP_EPOCHS with pure cross-entropy
  before mutual learning begins. This ensures peers start with a
  meaningful signal rather than random noise.

Output : __5_dml_online_distillation.json
         __dml_saved_models/ (peer checkpoints)
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  : {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU      : {torch.cuda.get_device_name(0)}", flush=True)
    cap = torch.cuda.get_device_capability()
    print(f"[init] Compute  : SM{cap[0]}{cap[1]}", flush=True)
    TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"[init] VRAM     : {TOTAL_VRAM_GB:.1f} GB", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__5_v2_dml_online_distillation.json"
SAVE_DIR              = "__dml_saved_models"

BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 32
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"
NUM_CLASSES  = 10
N_PEERS      = 2
TEMPERATURE  = 3.0   # τ for KL divergence
LR           = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4
USE_AMP      = DEVICE.type == "cuda"

WARMUP_EPOCHS = 2    # pure CE pre-training before mutual learning
KD_EPOCHS     = 15   # mutual learning epochs (matches spec)
TOTAL_EPOCHS  = WARMUP_EPOCHS + KD_EPOCHS

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] N peers      : {N_PEERS}", flush=True)
print(f"[init] τ={TEMPERATURE}  LR={LR}  AMP={USE_AMP}", flush=True)
print(f"[init] Warmup epochs: {WARMUP_EPOCHS}  KD epochs: {KD_EPOCHS}", flush=True)

[init] PyTorch  : 2.11.0+cu128
[init] Device   : cuda
[init] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[init] Compute  : SM120
[init] VRAM     : 8.0 GB
[init] N peers      : 2
[init] τ=3.0  LR=0.1  AMP=True
[init] Warmup epochs: 2  KD epochs: 15


In [6]:
# ── Model ──────────────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    """ResNet-50 with CIFAR-10 head: 3×3 conv1, no maxpool, 10-class fc."""
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


# ── Data ───────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_ld  = DataLoader(test_ds,  batch_size=512,        shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print(f"[data] Train batches: {len(train_ld)}  Test batches: {len(test_ld)}", flush=True)
    return train_ld, test_ld


# ── Helpers ────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    yp, yt = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall"   : float(recall_score(yt, yp, average="macro", zero_division=0)),
        "f1"       : float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

def measure_inference_detailed(model: nn.Module) -> dict:
    model = copy.deepcopy(model).eval().to(DEVICE)
    is_gpu = DEVICE.type == "cuda"

    # Single image
    dummy_single = torch.randn(1, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    # Batch 128
    dummy_batch = torch.randn(128, 3, 32, 32, device=DEVICE)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def compute_flops(model: nn.Module) -> float:
    try:
        from thop import profile as thop_profile
        m = copy.deepcopy(model).cpu().eval()
        dummy = torch.randn(1, 3, 32, 32)
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None

@torch.no_grad()
def evaluate_ensemble(peers: list, loader: DataLoader) -> float:
    for p in peers: p.eval()
    all_preds, all_labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        probs = torch.stack([
            F.softmax(p(inputs), dim=1).cpu() for p in peers
        ]).mean(0)
        all_preds.extend(probs.argmax(1).numpy())
        all_labels.extend(lbls.numpy())
    return float(accuracy_score(np.array(all_labels), np.array(all_preds)))

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Warm-start: train each peer independently with CE before mutual learning
# ══════════════════════════════════════════════════════════════════════════════
def warmup_peer(peer: nn.Module, optimizer, scaler, scheduler,
                train_loader: DataLoader, peer_idx: int) -> list:
    """Pre-train peer with pure cross-entropy for WARMUP_EPOCHS."""
    print(f"      [warmup] Peer {peer_idx} — {WARMUP_EPOCHS} CE epochs", flush=True)
    warmup_acc = []

    for epoch in range(WARMUP_EPOCHS):
        peer.train()
        correct = total = 0
        t_ep = time.perf_counter()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs  = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = peer(inputs)
                loss   = F.cross_entropy(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            correct += (logits.detach().argmax(1) == targets).sum().item()
            total   += targets.size(0)

        scheduler.step()
        acc = correct / total
        warmup_acc.append(round(acc, 6))
        ep_time = time.perf_counter() - t_ep
        print(f"      [warmup] Peer {peer_idx}  ep {epoch+1}/{WARMUP_EPOCHS} "
              f"acc={acc:.4f}  time={ep_time:.1f}s", flush=True)

    return warmup_acc


# ══════════════════════════════════════════════════════════════════════════════
# DML mutual learning phase
# ══════════════════════════════════════════════════════════════════════════════
def run_dml(train_loader: DataLoader, test_loader: DataLoader,
            baseline_acc: float) -> dict:

    print(f"\n  ┌─ [DML — N={N_PEERS}, τ={TEMPERATURE}]", flush=True)

    try:
        # ── Build peers ───────────────────────────────────────────────────────
        peers = [build_resnet50_cifar(NUM_CLASSES).to(DEVICE) for _ in range(N_PEERS)]
        print(f"      [init] {N_PEERS} × ResNet-50", flush=True)

        # Per-peer optimizers, schedulers, and scalers
        # (sharing one scaler across N backward passes causes scale collapse)
        optimizers = [
            torch.optim.SGD(p.parameters(), lr=LR, momentum=MOMENTUM,
                            weight_decay=WEIGHT_DECAY, nesterov=True)
            for p in peers
        ]
        # Total schedule spans warmup + KD epochs so cosine annealing is smooth
        schedulers = [
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TOTAL_EPOCHS)
            for opt in optimizers
        ]
        scalers = [torch.amp.GradScaler(enabled=USE_AMP) for _ in peers]

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        t_start = time.perf_counter()

        # ── Phase 1: Warm-start ───────────────────────────────────────────────
        print(f"\n  ── Phase 1 : Warm-start ({WARMUP_EPOCHS} epochs each peer) ───",
              flush=True)
        warmup_epoch_acc = []
        for i, (peer, opt, scaler, sch) in enumerate(
                zip(peers, optimizers, scalers, schedulers)):
            w_acc = warmup_peer(peer, opt, scaler, sch, train_loader, i)
            warmup_epoch_acc.append(w_acc)

        # ── Phase 2: Mutual learning (DML) ───────────────────────────────────
        print(f"\n  ── Phase 2 : DML mutual learning ({KD_EPOCHS} epochs) ─────────",
              flush=True)

        peer_epoch_acc = [[] for _ in range(N_PEERS)]

        for epoch in range(KD_EPOCHS):
            for p in peers: p.train()
            correct_per_peer = [0] * N_PEERS
            total = 0
            t_ep  = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                # ── Forward all peers ─────────────────────────────────────
                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    all_logits = [p(inputs) for p in peers]  # (B,C) each

                    # Detached soft probs (KL targets) — fully detached
                    all_probs_det = [
                        F.softmax(lg.detach() / TEMPERATURE, dim=1)
                        for lg in all_logits
                    ]

                # ── Per-peer backward (no retain_graph needed) ────────────
                for i in range(N_PEERS):
                    optimizers[i].zero_grad(set_to_none=True)

                    with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                        ce_i = F.cross_entropy(all_logits[i], targets)

                        # KL from every other peer (all detached)
                        kl_sum = torch.zeros(1, device=DEVICE)
                        for j in range(N_PEERS):
                            if j == i:
                                continue
                            log_pi = F.log_softmax(all_logits[i] / TEMPERATURE, dim=1)
                            kl_sum = kl_sum + F.kl_div(
                                log_pi, all_probs_det[j], reduction="batchmean"
                            ) * (TEMPERATURE ** 2)

                        kl_avg = kl_sum / max(N_PEERS - 1, 1)
                        loss_i = ce_i + kl_avg

                    scalers[i].scale(loss_i).backward()
                    scalers[i].step(optimizers[i])
                    scalers[i].update()

                with torch.no_grad():
                    for i in range(N_PEERS):
                        correct_per_peer[i] += (
                            all_logits[i].detach().argmax(1) == targets
                        ).sum().item()
                total += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    accs    = " | ".join(
                        f"p{i}={correct_per_peer[i]/total:.3f}" for i in range(N_PEERS)
                    )
                    print(f"      [DML ep {epoch+1}/{KD_EPOCHS}] "
                          f"batch {done}/{len(train_loader)}  {accs}  ETA={eta:.0f}s",
                          flush=True)

            for sch in schedulers: sch.step()
            sync()

            ep_time   = time.perf_counter() - t_ep
            remaining = KD_EPOCHS - (epoch + 1)
            acc_vals  = [round(correct_per_peer[i] / total, 6) for i in range(N_PEERS)]
            for i in range(N_PEERS):
                peer_epoch_acc[i].append(acc_vals[i])

            print(f"      [DML ep {epoch+1}/{KD_EPOCHS}] DONE  "
                  f"peer_accs={acc_vals}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s",
                  flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # ── Evaluate ──────────────────────────────────────────────────────────
        print("      [eval] Evaluating all peers ...", flush=True)
        peer_metrics = []
        for i, peer in enumerate(peers):
            m = evaluate(peer, test_loader)
            peer_metrics.append(m)
            print(f"             Peer {i}: acc={m['accuracy']:.4f}", flush=True)

        best_idx     = max(range(N_PEERS), key=lambda i: peer_metrics[i]["accuracy"])
        best_metrics = peer_metrics[best_idx]
        avg_acc      = float(np.mean([m["accuracy"] for m in peer_metrics]))

        print("      [eval] Ensemble ...", flush=True)
        ens_acc = evaluate_ensemble(peers, test_loader)
        print(f"             Ensemble: acc={ens_acc:.4f}", flush=True)

        # Detailed inference stats on the best peer
        inf = measure_inference_detailed(peers[best_idx])
        size_mb = model_size_mb(peers[best_idx])
        flops_G = compute_flops(peers[best_idx])
        total_params = param_count(peers[best_idx])
        acc_drop = baseline_acc - best_metrics["accuracy"]
        ens_drop = baseline_acc - ens_acc

        # ── Save peer checkpoints ─────────────────────────────────────────────
        os.makedirs(SAVE_DIR, exist_ok=True)
        saved_paths = []
        for i, peer in enumerate(peers):
            path = f"{SAVE_DIR}/dml_peer{i}_resnet50.pth"
            torch.save(peer.state_dict(), path)
            saved_paths.append(path)
            print(f"      [save] Peer {i} → {path}", flush=True)

        print(f"  └─ BestPeer={best_metrics['accuracy']:.4f}  "
              f"Ensemble={ens_acc:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            # ── Identification ────────────────────────────────────────────────
            "method"              : "online_distillation_dml",
            "variant"             : "DML",
            "model_arch"          : "ResNet-50 × 2 peers (CIFAR-10 modified, 3×3 conv1, no maxpool)",
            "dataset"             : "CIFAR-10",
            # ── Hyperparameters ───────────────────────────────────────────────
            "n_peers"             : N_PEERS,
            "temperature"         : TEMPERATURE,
            "alpha"               : "N/A",     # DML uses equal CE + KL, no α weighting
            "lr"                  : LR,
            "momentum"            : MOMENTUM,
            "weight_decay"        : WEIGHT_DECAY,
            "lr_schedule"         : "cosine",
            "warmup_epochs"       : WARMUP_EPOCHS,
            "kd_epochs"           : KD_EPOCHS,
            "total_epochs"        : TOTAL_EPOCHS,
            "train_device"        : str(DEVICE),
            "use_amp"             : USE_AMP,
            # ── Training stats ────────────────────────────────────────────────
            "train_time_s"        : round(train_time_s, 2),
            "warmup_epoch_acc"    : warmup_epoch_acc,   # [peer0_list, peer1_list]
            "peer_epoch_acc"      : peer_epoch_acc,     # [peer0_list, peer1_list]
            "peer_final_acc"      : [m["accuracy"] for m in peer_metrics],
            "best_peer_idx"       : best_idx,
            "peak_gpu_memory_mb"  : peak_gpu_mb,
            # ── Standardised metrics block (mirrors response-based structure) ──
            "accuracy"            : round(best_metrics["accuracy"],  6),
            "precision"           : round(best_metrics["precision"], 6),
            "recall"              : round(best_metrics["recall"],    6),
            "f1"                  : round(best_metrics["f1"],        6),
            "accuracy_drop"       : round(acc_drop, 6),
            "ensemble_accuracy"   : round(ens_acc, 6),
            "ensemble_drop"       : round(ens_drop, 6),
            "avg_peer_accuracy"   : round(avg_acc, 6),
            # ── Model size / efficiency ────────────────────────────────────────
            "params"              : total_params,
            "params_nonzero"      : total_params,
            "sparsity_actual"     : 0.0,
            "size_mb"             : round(size_mb, 4),
            "flops_G"             : flops_G,
            "inference_ms"        : inf,
            # ── Saved models ──────────────────────────────────────────────────
            "saved_model_paths"   : saved_paths,
            # ── Description ───────────────────────────────────────────────────
            "description"         : (
                f"Online KD (DML, N={N_PEERS}, τ={TEMPERATURE}, "
                f"warm-start={WARMUP_EPOCHS}ep, cosine, "
                f"kd_epochs={KD_EPOCHS}, amp={USE_AMP})"
            ),
            "status"              : "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "method": "online_distillation_dml", "variant": "DML",
            "n_peers": N_PEERS, "temperature": TEMPERATURE,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }



In [8]:
# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Online Distillation — DML (Deep Mutual Learning)")
    print(f"  Models : ResNet-50 × {N_PEERS} peers — no pre-trained teacher")
    print(f"  Device : {DEVICE}  |  Batch : {BATCH_SIZE}  |  AMP : {USE_AMP}")
    print(f"  τ={TEMPERATURE}  LR={LR}  Warmup={WARMUP_EPOCHS}ep  KD={KD_EPOCHS}ep")
    print(f"{'='*60}\n")

    # ── Baseline ──────────────────────────────────────────────────────────────
    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    # ── Reference model info ──────────────────────────────────────────────────
    ref = build_resnet50_cifar(NUM_CLASSES)
    ref_size_mb  = model_size_mb(ref)
    ref_params   = param_count(ref)
    print(f"  Per-peer — Size: {ref_size_mb:.2f} MB  Params: {ref_params:,}\n",
          flush=True)

    train_loader, test_loader = get_loaders()

    # ── Run DML ───────────────────────────────────────────────────────────────
    result = run_dml(train_loader, test_loader, baseline_acc)

    # ── Report (mirrors response-based structure) ──────────────────────────────
    report = {
        "experiment"      : "online_distillation_dml",
        "method"          : "online_distillation_dml",
        "variant"         : "DML",
        "description"     : (
            "Online Distillation — DML: two ResNet-50 peers co-trained "
            "with mutual KL divergence, warm-start init, no pre-trained teacher."
        ),
        "model_arch"      : "ResNet-50 × 2 (CIFAR-10 modified, 3×3 conv1, no maxpool)",
        "dataset"         : "CIFAR-10",
        "train_device"    : str(DEVICE),
        "use_amp"         : USE_AMP,
        "batch_size"      : BATCH_SIZE,
        "temperature"     : TEMPERATURE,
        "n_peers"         : N_PEERS,
        "warmup_epochs"   : WARMUP_EPOCHS,
        "kd_epochs"       : KD_EPOCHS,
        "baseline"        : baseline,
        "model_info"      : {
            "size_mb" : round(ref_size_mb, 4),
            "params"  : ref_params,
        },
        "best_config"     : {
            "variant"          : "DML",
            "n_peers"          : N_PEERS,
            "temperature"      : TEMPERATURE,
            "accuracy"         : result.get("accuracy"),
            "ensemble_accuracy": result.get("ensemble_accuracy"),
            "accuracy_drop"    : result.get("accuracy_drop"),
            "size_mb"          : result.get("size_mb"),
            "inference_ms"     : result.get("inference_ms"),
        },
        "results"         : [result],
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if result.get("accuracy") is not None:
        print(f"  Best peer acc    : {result['accuracy']:.4f}  "
              f"(drop={result['accuracy_drop']:+.4f})")
        print(f"  Ensemble acc     : {result['ensemble_accuracy']:.4f}  "
              f"(drop={result['ensemble_drop']:+.4f})")
        print(f"  Avg peer acc     : {result['avg_peer_accuracy']:.4f}")
        inf = result["inference_ms"]
        print(f"  Inference        : single={inf['single_img_gpu_ms']:.2f}ms  "
              f"batch128={inf['batch128_gpu_ms']:.2f}ms  "
              f"throughput={inf['throughput_imgs_sec']:.0f}imgs/s")
        print(f"  Model size       : {result['size_mb']:.2f} MB")
        print(f"  Saved models     : {result['saved_model_paths']}")


if __name__ == "__main__":
    main()


  Online Distillation — DML (Deep Mutual Learning)
  Models : ResNet-50 × 2 peers — no pre-trained teacher
  Device : cuda  |  Batch : 128  |  AMP : True
  τ=3.0  LR=0.1  Warmup=2ep  KD=15ep

  Baseline accuracy : 0.9320

  Per-peer — Size: 90.03 MB  Params: 23,520,842

[data] Train batches: 391  Test batches: 20

  ┌─ [DML — N=2, τ=3.0]
      [init] 2 × ResNet-50

  ── Phase 1 : Warm-start (2 epochs each peer) ───
      [warmup] Peer 0 — 2 CE epochs
      [warmup] Peer 0  ep 1/2 acc=0.1115  time=44.7s
      [warmup] Peer 0  ep 2/2 acc=0.1756  time=48.7s
      [warmup] Peer 1 — 2 CE epochs
      [warmup] Peer 1  ep 1/2 acc=0.1864  time=49.9s
      [warmup] Peer 1  ep 2/2 acc=0.2910  time=50.0s

  ── Phase 2 : DML mutual learning (15 epochs) ─────────
      [DML ep 1/15] batch 100/391  p0=0.224 | p1=0.339  ETA=61s
      [DML ep 1/15] batch 200/391  p0=0.244 | p1=0.349  ETA=40s
      [DML ep 1/15] batch 300/391  p0=0.262 | p1=0.357  ETA=19s
      [DML ep 1/15] DONE  peer_accs=[0.27528, 